SparkSession + catálogo Iceberg

In [7]:
import os
from pyspark.sql import SparkSession

# Usa la ruta Ivy tradicional y prepara subcarpetas para resolver dependencias
ivy_dir = "/home/jovyan/.ivy2"
os.makedirs(f"{ivy_dir}/cache", exist_ok=True)
os.makedirs(f"{ivy_dir}/jars", exist_ok=True)

spark = (
    SparkSession.builder
    .appName("bronze-payments-stream")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
        ])
    )
    .config("spark.jars.ivy", ivy_dir)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "admin123")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.bronze")

DataFrame[]

Lectura Kafka + esquema original

In [8]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("payment_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("ip", StringType(), True),
    StructField("country", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("mcc", StringType(), True),
])

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "payment-events")
    .option("startingOffsets", "latest")
    .load()
)

bronze_df = (
    kafka_df
    .selectExpr("CAST(value AS STRING) AS json_payload")
    .select(from_json(col("json_payload"), schema).alias("data"))
    .select("data.*")
)

Escritura streaming a Iceberg Bronze

In [9]:
import time

query = (
    bronze_df.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/chk/bronze_payments_raw")
    .toTable("lakehouse.bronze.payments_raw")
)

print(f"✅ Streaming query iniciado")
print(f"ID Query: {query.id if query else 'N/A'}")
print(f"Nombre: {query.name if query else 'N/A'}")
print(f"Status: {query.status if query else 'No iniciado'}")

# Dejar correr 10 segundos para que ingieran algunos eventos
time.sleep(10)

# Mostrar status
if query:
    print(f"\nStatus después de 10s:")
    print(f"  Activo: {query.isActive}")
    print(f"  Estatísticas: {query.lastProgress}")


✅ Streaming query iniciado
ID Query: c17f9d42-a05d-4067-a339-4ee159c33c01
Nombre: None
Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}

Status después de 10s:
  Activo: True
  Estatísticas: {'id': 'c17f9d42-a05d-4067-a339-4ee159c33c01', 'runId': 'c574847f-0fd1-44fa-a9a4-dbea3058b072', 'name': None, 'timestamp': '2026-03-24T16:05:01.399Z', 'batchId': 25, 'numInputRows': 1, 'inputRowsPerSecond': 5.0, 'processedRowsPerSecond': 4.878048780487805, 'durationMs': {'addBatch': 164, 'commitOffsets': 15, 'getBatch': 1, 'latestOffset': 3, 'queryPlanning': 8, 'triggerExecution': 205, 'walCommit': 14}, 'stateOperators': [], 'sources': [{'description': 'KafkaV2[Subscribe[payment-events]]', 'startOffset': {'payment-events': {'0': 100}}, 'endOffset': {'payment-events': {'0': 101}}, 'latestOffset': {'payment-events': {'0': 101}}, 'numInputRows': 1, 'inputRowsPerSecond': 5.0, 'processedRowsPerSecond': 4.878048780487805, 'metrics': {'avgOffsetsBehindLatest'

Verificación rápida

In [10]:
# 🔍 DEBUG: Verifica que Kafka tiene eventos
print("=" * 80)
print("🔍 DEBUGGING KAFKA")
print("=" * 80)

# Leer eventos raw desde Kafka (sin transformar)
kafka_raw = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "payment-events")
    .option("startingOffsets", "earliest")
    .load()
)

print("\n📨 Total de eventos en Kafka:")
count = kafka_raw.count()
print(f"   Cantidad: {count}")

if count > 0:
    print("\n🔎 Primeros 3 eventos raw:")
    kafka_raw.limit(3).select(
        "key",
        "value",
        "partition",
        "offset",
        "timestamp"
    ).show(truncate=False)
else:
    print("   ⚠️  No hay eventos en Kafka. Verifica que generar_datos.py está ejecutándose.")


🔍 DEBUGGING KAFKA

📨 Total de eventos en Kafka:
   Cantidad: 102

🔎 Primeros 3 eventos raw:
+-------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
# ✅ VERIFICACIÓN FINAL
print("=" * 80)
print("✅ VERIFICACIÓN FINAL DEL PROCESO")
print("=" * 80)

# 1. Estado del streaming query
print("\n📡 Estado del Streaming Query:")
if 'query' in globals():
    print(f"   Activo: {query.isActive}")
    print(f"   ID: {query.id}")
    if query.isActive:
        print(f"   ✓ El streaming está CORRIENDO en background")
    else:
        print(f"   ✗ El streaming NO está activo (puede haber terminado o error)")
else:
    print("   ✗ No hay query definitida. Ejecuta la celda 3 primero.")

# 2. Verificar tabla Iceberg
print("\n📊 Contenido de la tabla Iceberg (lakehouse.bronze.payments_raw):")
try:
    total = spark.sql("SELECT COUNT(*) as total FROM lakehouse.bronze.payments_raw").collect()[0]["total"]
    print(f"   Total registros: {total}")
    
    if total > 0:
        print(f"\n   ✓ ¡ÉXITO! Se están ingiriendo eventos. Ultimos 3:")
        spark.sql("""
            SELECT event_time, payment_id, customer_id, status 
            FROM lakehouse.bronze.payments_raw 
            ORDER BY event_time DESC 
            LIMIT 3
        """).show(truncate=False)
    else:
        print("   ✗ La tabla está vacía. Revisa:")
        print("      1. ¿El generador está corriendo? (generar_datos.py)")
        print("      2. ¿Hay eventos en Kafka? (ejecuta la celda de DEBUG)")
        print("      3. ¿El parsing del JSON es correcto?")
except Exception as e:
    print(f"   ✗ Error al consultar tabla: {e}")


✅ VERIFICACIÓN FINAL DEL PROCESO

📡 Estado del Streaming Query:
   Activo: True
   ID: c17f9d42-a05d-4067-a339-4ee159c33c01
   ✓ El streaming está CORRIENDO en background

📊 Contenido de la tabla Iceberg (lakehouse.bronze.payments_raw):
   Total registros: 37

   ✓ ¡ÉXITO! Se están ingiriendo eventos. Ultimos 3:
+---------------------------+------------------------------------+-----------+--------+
|event_time                 |payment_id                          |customer_id|status  |
+---------------------------+------------------------------------+-----------+--------+
|2026-03-24T16:05:01.912527Z|9ff20bd7-a4a7-46ff-b725-d23a181d0edb|CUST_00042 |approved|
|2026-03-24T16:05:01.811232Z|79c79fd4-33cf-4255-93be-ae22644dd740|CUST_00039 |approved|
|2026-03-24T16:05:01.509959Z|80d0e928-fcbd-42db-bd7e-01494eb2a2d7|CUST_00040 |approved|
+---------------------------+------------------------------------+-----------+--------+

